# 🏀⚽ Sports Classifier
## Image Classification with fastai — *Basketball vs Soccer*

> *Adapting the Chapter 2 Bear Classifier from*
> *"Deep Learning for Coders with fastai and PyTorch"*

This notebook trains a binary image classifier to predict:
**BASKETBALL** 🏀 or **SOCCER** ⚽?

We follow the **identical 8-step workflow** as the Bear Classifier — only the domain changes.

| Step | Task | Key fastai concept |
|:----:|------|-------------------|
| 1 | 📥 Gather Data | DuckDuckGo + `download_images` |
| 2 | 🧹 Clean Data | `verify_images`, `Path.unlink` |
| 3 | 🗂️ Build DataLoaders | `DataBlock`, `parent_label`, `RandomSplitter` |
| 4 | 🏋️ Train Model | `vision_learner`, `resnet18`, `fine_tune` |
| 5 | 📊 Confusion Matrix | `ClassificationInterpretation` |
| 6 | 🔍 Top Losses | `plot_top_losses` |
| 7 | 💾 Export | `learn.export()` |
| 8 | 🔮 Predict | `load_learner`, `learn.predict` |

---

### ⚠️ Limitations

This is a **binary image classifier** trained on web-scraped photos — not a professional
sports analytics system. Hard examples and unusual camera angles can still confuse the model.
This is a **learning project** demonstrating fastai. 🎓

---
## 🛠️ Step 0: Install & Import

Uncomment the pip line if you haven't installed the packages yet.

In [ ]:
# Uncomment to install (run once):
# !pip install fastai ddgs requests -q

In [ ]:
from fastai.vision.all import *
from ddgs import DDGS
import time, shutil, os, random
from pathlib import Path

# Change working dir to project root so data/ and models/ resolve correctly
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

PROJECT_ROOT = Path.cwd()
DATA_DIR     = PROJECT_ROOT / 'data'
MODELS_DIR   = PROJECT_ROOT / 'models'

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Models dir   : {MODELS_DIR}')

---
## 📥 Step 1: Gather Data

We search DuckDuckGo with **multiple queries per category** to get diverse images:
different settings (classroom, board games), age groups, and image styles.

| Category | What we're looking for |
|---|---|
| `basketball` | Players dribbling, shooting, game action, basketball courts |
| `soccer` | Players kicking, passing, match scenes, soccer fields |

Target: ~150–200 images per category.

In [ ]:
def search_images(term: str, max_images: int = 100) -> list[str]:
    """Search DuckDuckGo for images with retry/backoff for rate limits."""
    print(f"  Searching: '{term}'")
    for attempt in range(4):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.images(term, max_results=max_images))
            urls = [r['image'] for r in results if r.get('image')]
            print(f"    → {len(urls)} URLs found")
            return urls
        except Exception:
            wait = 8 * (2 ** attempt) + random.uniform(1, 4)
            print(f"    ⏳ Rate limited, retrying in {wait:.0f}s…")
            time.sleep(wait)
    return []

# Quick sanity check
test = search_images('basketball player shooting ball', max_images=3)
print(f'\nSample URL: {test[0] if test else "none"}')

In [ ]:
# ── Download BASKETBALL images ───────────────────────────────────────────────
basketball_queries = [
    'basketball player dribbling',
    'basketball game action shot',
    'basketball player shooting hoop',
    'nba basketball court game',
]

basketball_dir = DATA_DIR / 'basketball'
basketball_dir.mkdir(parents=True, exist_ok=True)

print('Downloading BASKETBALL images…')
print('=' * 40)
for query in basketball_queries:
    urls = search_images(query, max_images=50)
    download_images(basketball_dir, urls=urls)
    print(f"  ✓ Saved images for: '{query}'")
    time.sleep(7)

print(f'\nTotal basketball images: {len(get_image_files(basketball_dir))}')

In [ ]:
# ── Download SOCCER images ───────────────────────────────────────────────────
soccer_queries = [
    'soccer player kicking ball',
    'soccer match action photo',
    'football player dribbling field',
    'soccer goal celebration',
]

soccer_dir = DATA_DIR / 'soccer'
soccer_dir.mkdir(parents=True, exist_ok=True)

print('Downloading SOCCER images…')
print('=' * 40)
time.sleep(20)  # cool down between categories
for query in soccer_queries:
    urls = search_images(query, max_images=50)
    download_images(soccer_dir, urls=urls)
    print(f"  ✓ Saved images for: '{query}'")
    time.sleep(7)

print(f'\nTotal soccer images: {len(get_image_files(soccer_dir))}')

In [ ]:
n_basketball = len(get_image_files(basketball_dir))
n_soccer     = len(get_image_files(soccer_dir))
print('📊 Download Summary')
print(f'  basketball : {n_basketball:4d} images')
print(f'  soccer     : {n_soccer:4d} images')
print(f'  total      : {n_basketball + n_soccer:4d} images')

# Preview a sample
sample = get_image_files(basketball_dir)[:3] + get_image_files(soccer_dir)[:3]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, fp in zip(axes.flat, sample):
    try:
        ax.imshow(PILImage.create(fp).resize((300, 200)))
        ax.set_title(fp.parent.name, fontsize=11, fontweight='bold')
    except Exception:
        ax.set_visible(False)
    ax.axis('off')
plt.suptitle('Sample downloaded images (before cleaning)', fontsize=13)
plt.tight_layout(); plt.show()

---
## �� Step 2: Verify and Clean the Dataset

Web-scraped images are messy — truncated downloads, HTML error pages, zero-byte files.
`verify_images()` tries to open each file with Pillow and returns the ones that fail.

In [ ]:
print('🧹 Cleaning dataset…\n')
total_removed = 0
for folder, name in [(basketball_dir, 'basketball'), (soccer_dir, 'soccer')]:
    files = get_image_files(folder)
    print(f'  {name}: checking {len(files)} images…')
    failed = verify_images(files)
    failed.map(Path.unlink)
    total_removed += len(failed)
    print(f'    {"✗ Removed " + str(len(failed)) + " corrupt images" if failed else "✓ All valid"}')
print(f'\n  Total removed: {total_removed}')

In [ ]:
n_basketball = len(get_image_files(basketball_dir))
n_soccer     = len(get_image_files(soccer_dir))
print('📊 Post-cleaning Counts')
print(f'  basketball : {n_basketball:4d}')
print(f'  soccer     : {n_soccer:4d}')
print(f'  total      : {n_basketball + n_soccer:4d}')
print('\n✅ Ready for training!' if n_basketball >= 30 and n_soccer >= 30
     else '\n⚠️ Very few images — re-run download cells.')

---
## 🗂️ Step 3: Build DataLoaders

The **DataBlock API** is a declarative recipe for building a training pipeline:

```
DataBlock(
    blocks     = (ImageBlock, CategoryBlock),   # inputs + outputs
    get_items  = get_image_files,               # how to find data
    get_y      = parent_label,                  # folder name = label
    splitter   = RandomSplitter(0.2, seed=42),  # 80/20 train/val
    item_tfms  = RandomResizedCrop(224),        # resize + crop augment
    batch_tfms = aug_transforms(),              # flip, rotate, colour
)
```

`parent_label` means the folder name becomes the label:
- `data/basketball/img.jpg` → `"basketball"`
- `data/soccer/img.jpg` → `"soccer"`

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms(),
).dataloaders(DATA_DIR, bs=32)

print(f'Classes (vocab) : {dls.vocab}')
print(f'Training images : {len(dls.train_ds)}')
print(f'Validation imgs : {len(dls.valid_ds)}')

In [ ]:
print('Sample training batch (with augmentations):')
dls.show_batch(max_n=9, nrows=3, figsize=(12, 9))

---
## 🏋️ Step 4: Train the Model

**Transfer learning** from ResNet18 pretrained on ImageNet — it already understands
edges, textures, faces, and objects. We fine-tune it to tell apart basketball vs soccer.

`fine_tune(4)` trains for 5 epochs:
- **Epoch 0** — new classification head only (backbone frozen)
- **Epochs 1–4** — all layers, discriminative learning rates

In [ ]:
learn = vision_learner(dls, resnet18, metrics=accuracy)
print(f'Backbone       : resnet18 (pretrained ImageNet)')
print(f'Output classes : {dls.vocab}')

In [ ]:
learn.fine_tune(4)

---
## 📊 Step 5: Evaluate the Model

We use `ClassificationInterpretation` to understand mistakes:

1. **Confusion matrix** — which class gets confused with which?
2. **Top losses** — the individual images the model was most wrong about

Top losses are the most actionable: they show mislabelled images and ambiguous cases.

In [ ]:
val_loss, val_acc = learn.validate()
print(f'Validation Loss     : {val_loss:.4f}')
print(f'Validation Accuracy : {val_acc:.4f}  ({val_acc * 100:.1f}%)')

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)

# Confusion Matrix — off-diagonal cells are mistakes
# For our problem:
#   basketball predicted as soccer → basketball miss
#   soccer predicted as basketball → soccer miss
print('Confusion Matrix (rows=actual, cols=predicted):')
interp.plot_confusion_matrix(figsize=(5, 4))

In [ ]:
# Top Losses — caption: predicted / actual / loss / probability
# High loss = model was very confident BUT very wrong.
# These are your best candidates for mislabelled or ambiguous images.
print('Top 9 Losses:')
interp.plot_top_losses(9, nrows=3, figsize=(13, 11))

---
## 🔍 Step 6: Clean with Model Assistance

> *"Train first, then use the model's mistakes to guide cleaning."* — fastai book

`ImageClassifierCleaner` shows validation images sorted by loss and lets you:
- 🗑️ **Delete** off-topic images (cartoons, stock photos, ads)
- 🔀 **Move** mislabelled images to the correct folder

After selections, run the cell below to apply them.

In [ ]:
cleaner = ImageClassifierCleaner(learn)
cleaner

In [ ]:
deleted = moved = 0
for idx in cleaner.delete():
    cleaner.fns[idx].unlink()
    deleted += 1
for idx, cat in cleaner.change():
    shutil.move(str(cleaner.fns[idx]), DATA_DIR / cat)
    moved += 1
print(f'Deleted : {deleted}  |  Moved : {moved}')
if deleted + moved > 0:
    print('💡 Re-run steps 3–4 to retrain on the cleaned dataset.')

In [ ]:
# Retrain after cleaning — uncomment and run:
# dls   = DataBlock(
#     blocks=(ImageBlock, CategoryBlock),
#     get_items=get_image_files,
#     get_y=parent_label,
#     splitter=RandomSplitter(valid_pct=0.2, seed=42),
#     item_tfms=RandomResizedCrop(224, min_scale=0.5),
#     batch_tfms=aug_transforms(),
# ).dataloaders(DATA_DIR, bs=32)
# learn = vision_learner(dls, resnet18, metrics=accuracy)
# learn.fine_tune(4)

---
## 💾 Step 7: Export the Model

`learn.export()` saves everything to one `.pkl` file:
weights + architecture + vocabulary + preprocessing transforms.
Load it on any machine without rebuilding the DataBlock.

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
model_path = MODELS_DIR / 'sport_classifier.pkl'
learn.export(model_path)
print(f'✅ Exported to : {model_path}')
print(f'   Size        : {model_path.stat().st_size / 1_048_576:.1f} MB')

---
## 🔮 Step 8: Run Inference

Load the model from disk — no DataBlock needed. Then call `learn.predict()` on any image.

In [ ]:
learn_inf = load_learner(MODELS_DIR / 'sport_classifier.pkl')
print(f'Classes : {learn_inf.dls.vocab}')

In [ ]:
img_path = get_image_files(DATA_DIR)[0]
img      = PILImage.create(img_path)
label, index, probs = learn_inf.predict(img)

print(f'Image      : {img_path.name}')
print(f'Prediction : {label}')
print(f'Confidence : {float(probs[index]) * 100:.1f}%')
print()
for cls, p in zip(learn_inf.dls.vocab, probs):
    bar = '█' * int(float(p) * 40)
    print(f'  {cls:<16s}: {float(p)*100:5.1f}%  {bar}')

In [ ]:
# Visual prediction grid over the first 6 images
test_files = get_image_files(DATA_DIR)[:6]
fig, axes  = plt.subplots(2, 3, figsize=(13, 8))
for ax, fp in zip(axes.flat, test_files):
    img = PILImage.create(fp)
    pred, pred_idx, pred_probs = learn_inf.predict(img)
    actual  = fp.parent.name
    conf    = float(pred_probs[pred_idx]) * 100
    correct = pred == actual
    ax.imshow(img.resize((300, 200)))
    ax.set_title(f"{'✅' if correct else '❌'} {pred} ({conf:.0f}%)\nactual={actual}",
                 fontsize=9, color='green' if correct else 'red', fontweight='bold')
    ax.axis('off')
plt.suptitle('Model predictions on sample images', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Command-line equivalent:
#   python src/predict.py data/basketball/img_0001_abc12345.jpg
demo = get_image_files(DATA_DIR)[0]
print(f'python src/predict.py {demo}')

---
## 🎉 What We Built

The identical 8-step fastai pipeline, new domain:

| Step | What we did |
|------|-------------|
| **Data** | Scraped ~300 images via DuckDuckGo (basketball + soccer) |
| **Cleaning** | Removed corrupt files with `verify_images` |
| **DataLoaders** | `DataBlock` with `parent_label` folder convention |
| **Training** | Fine-tuned ResNet18 via transfer learning |
| **Evaluation** | Confusion matrix + top-losses |
| **Cleaning v2** | `ImageClassifierCleaner` widget |
| **Export** | Self-contained `.pkl` file |
| **Inference** | `load_learner` → `predict` |

### 🚀 Ideas to go further
- Larger backbone: `resnet34`, `convnext_small`
- More epochs: `fine_tune(8)`
- Learning rate finder: `learn.lr_find()`
- More data: manual curation from sports image sources
- Deploy with Gradio for a web demo